# 01 - Analyse exploratoire

Generation du jeu de donnees synthetique, controles qualite, distribution des groupes proteges et detection de proxies.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

for path in ['data/processed', 'models', 'outputs', 'reports', 'reports/figures']:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_processing import DataProcessor, generate_sample_data

DATA_PATH = PROJECT_ROOT / 'data/processed/dataset.csv'
df = generate_sample_data(n_samples=5000, random_state=42)
df.to_csv(DATA_PATH, index=False)
df.head()

,gender,race,age,education,years_experience,test_score,label
0,Male,White,49,Bachelor,26,88.931167,1
1,Female,White,53,Bachelor,3,90.189917,1
2,Female,Asian,67,High School,0,87.504444,1
3,Male,White,54,Master,0,80.106919,1
4,Male,Asian,65,PhD,28,76.915806,0


In [3]:
processor = DataProcessor(protected_attributes=['gender', 'race'], label_name='label')
demographics = processor.explore_demographics(df)
quality = processor.check_data_quality(df)
proxies = processor.identify_proxies(df, threshold=0.3)

print('Dataset:', df.shape)
print('Valeurs manquantes:', sum(quality['missing_values'].values()))
print('Doublons:', quality['duplicates'])
print('Proxies detectes:', {k: len(v) for k, v in proxies.items()})

display(pd.DataFrame({
    attr: demographics[attr]['proportions'] for attr in ['gender', 'race']
}).fillna(''))

Dataset: (5000, 7)
Valeurs manquantes: 0
Doublons: 0
Proxies detectes: {'gender': 0, 'race': 0}


,gender,race
Male,0.5998,
Female,0.4002,
White,,0.6218
Black,,0.1856
Hispanic,,0.0964
Asian,,0.0962


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, attr in zip(axes, ['gender', 'race']):
    sns.countplot(data=df, x=attr, hue='label', ax=ax)
    ax.set_title(f'Distribution label par {attr}')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
fig.savefig(PROJECT_ROOT / 'reports/figures/eda_demographics.png', dpi=150, bbox_inches='tight')
plt.close(fig)